# 00 Setup and download

**Owner:** Sahil

Creates the folder structure, locates the four raw sources, prints the schema and 5 sample rows of
every raw file, checks the columns later notebooks depend on, and downloads one ClubElo snapshot per
edition date (cached to `data/raw/clubelo/`).

**Output:** an inventory table (file, rows, columns, date range) in `reports/data_quality.md`, section `00`.

Run top to bottom on a fresh kernel. ClubElo snapshots that are already cached are never re-downloaded.

In [1]:
import sys
import logging
from pathlib import Path

# Find the repo root by walking up from the working directory (no absolute paths).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import config, dq, clubelo

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("Repo root:", ROOT.name, "| pandas", pd.__version__, "| python", sys.version.split()[0])

CHECKS = []  # (check, passed, detail)
def check(name, passed, detail=""):
    CHECKS.append({"Check": name, "Result": "PASS" if passed else "FAIL", "Detail": detail})
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + detail if detail else ""))

FLAGS = []  # issues later notebooks must handle
INVENTORY = []

Repo root: career-mode-promise | pandas 3.1.1 | python 3.11.13


## 1. Folder structure and log files

In [2]:
for d in config.ALL_DIRS:
    d.mkdir(parents=True, exist_ok=True)

if not config.DECISIONS.exists():
    config.DECISIONS.write_text(
        "# Decisions\n\nEvery judgement call: date, decision, reason, alternatives considered.\n",
        encoding="utf-8")
if not config.DATA_QUALITY.exists():
    config.DATA_QUALITY.write_text(dq._HEADER, encoding="utf-8")

for d in config.ALL_DIRS:
    print(f"{str(d.relative_to(ROOT)):28s} exists={d.exists()}")

data/raw/fifa                exists=True
data/raw/transfermarkt       exists=True
data/raw/clubelo             exists=True
data/raw/fifa_ranking        exists=True
data/manual                  exists=True
data/clean                   exists=True
notebooks                    exists=True
outputs/charts               exists=True
reports                      exists=True


## 2. Locate the raw sources

In [3]:
expected = {
    "fifa": [config.FIFA_PLAYERS],
    "transfermarkt": [config.RAW_TM / f"{t}.csv" for t in config.TM_REQUIRED_TABLES],
    "fifa_ranking": [config.RANKING_FILE],
    "club_elo_mirror": [config.RAW_CLUBELO_MIRROR / "EloRatings.csv", config.RAW_CLUBELO_MIRROR / "Matches.csv"],
}
missing = {src: [p.name for p in paths if not p.exists()] for src, paths in expected.items()}
missing = {k: v for k, v in missing.items() if v}
for src, files in missing.items():
    print(f"MISSING in data/raw/{src}/: {files}\n  download with: {config.KAGGLE_COMMANDS[src]}")
check("All Kaggle source files present", not missing, "; ".join(f"{k}: {v}" for k, v in missing.items()))
assert not missing, "Download the missing sources (commands above), then re-run."

PASS - All Kaggle source files present 


## 3. Helpers: schema, samples and inventory rows

In [4]:
def schema(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

def describe(label, df):
    print(f"\n=== {label}: {len(df):,} rows x {df.shape[1]} columns")
    with pd.option_context("display.max_rows", 200, "display.max_colwidth", 60):
        display(schema(df))
        display(df.head(5))

def date_range(series, kind="date"):
    if kind == "season":
        s = pd.to_numeric(series, errors="coerce").dropna()
        return f"seasons {int(s.min())}–{int(s.max())}" if len(s) else ""
    s = pd.to_datetime(series, errors="coerce", format="mixed")
    bad = int(series.notna().sum() - s.notna().sum())
    rng = f"{s.min():%Y-%m-%d} to {s.max():%Y-%m-%d}"
    if kind == "dob":
        rng = "DOB " + rng
    if bad:
        rng += f" ({bad:,} unparseable)"
    return rng

def add_inventory(source, path, df, rng="", note=""):
    INVENTORY.append({"Source": source, "File": str(path.relative_to(config.RAW)),
                      "Rows": len(df), "Columns": df.shape[1], "Date range": rng, "Notes": note})

## 4. FIFA (EA Sports FC 24 Complete Player Dataset)
Only `male_players.csv` is used in the analysis; the teams and coaches files are listed for completeness.

In [5]:
fifa = pd.read_csv(config.FIFA_PLAYERS, low_memory=False)
describe("male_players.csv", fifa)

per_ed = (fifa.groupby("fifa_version")
              .agg(rows=("player_id", "size"),
                   n_dates=("update_as_of", "nunique"),
                   date=("update_as_of", "first"),
                   n_updates=("fifa_update", "nunique"))
              .rename(index=lambda v: int(v)))
per_ed["label"] = per_ed.index.map(config.EDITION_LABELS)
per_ed["expected_date"] = per_ed.index.map(config.EXPECTED_EDITION_DATES)
display(per_ed)

check("FIFA: one snapshot date per edition", bool((per_ed["n_dates"] == 1).all()))
check("FIFA: editions are exactly 15–24", sorted(per_ed.index) == config.EDITIONS)
check("FIFA: edition dates match CLAUDE.md section 7",
      bool((per_ed["date"] == per_ed["expected_date"]).all()))

# Working edition dates come from the data, not from the constants.
EDITION_DATES = per_ed["date"].to_dict()

add_inventory("FIFA", config.FIFA_PLAYERS, fifa, date_range(fifa["update_as_of"]),
              f"{per_ed.shape[0]} editions (15–24), one launch snapshot each; {fifa['player_id'].nunique():,} unique player_id")


=== male_players.csv: 180,021 rows x 109 columns


,dtype,non_null,null_%,example
player_id,int64,180021,0.0,231747
player_url,str,180021,0.0,/player/231747/kylian-mbappe/240002
fifa_version,float64,180021,0.0,24.0
fifa_update,float64,180021,0.0,2.0
update_as_of,str,180021,0.0,2023-09-22
short_name,str,180021,0.0,K. Mbappé
long_name,str,180021,0.0,Kylian Mbappé Lottin
player_positions,str,180021,0.0,"ST, LW"
overall,int64,180021,0.0,91
potential,int64,180021,0.0,94


,player_id,player_url,fifa_version,fifa_update,update_as_of,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,dob,height_cm,weight_kg,club_team_id,club_name,league_id,league_name,league_level,club_position,club_jersey_number,club_loaned_from,club_joined_date,club_contract_valid_until_year,nationality_id,nationality_name,nation_team_id,nation_position,nation_jersey_number,preferred_foot,weak_foot,skill_moves,international_reputation,work_rate,body_type,real_face,release_clause_eur,player_tags,player_traits,pace,shooting,passing,dribbling,defending,physic,attacking_crossing,attacking_finishing,attacking_heading_accuracy,attacking_short_passing,attacking_volleys,skill_dribbling,skill_curve,skill_fk_accuracy,skill_long_passing,skill_ball_control,movement_acceleration,movement_sprint_speed,movement_agility,movement_reactions,movement_balance,power_shot_power,power_jumping,power_stamina,power_strength,power_long_shots,mentality_aggression,mentality_interceptions,mentality_positioning,mentality_vision,mentality_penalties,mentality_composure,defending_marking_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_diving,goalkeeping_handling,goalkeeping_kicking,goalkeeping_positioning,goalkeeping_reflexes,goalkeeping_speed,ls,st,rs,lw,lf,cf,rf,rw,lam,cam,ram,lm,lcm,cm,rcm,rm,lwb,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
0,231747,/player/231747/kylian-mbappe/240002,24.0,2.0,2023-09-22,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,94,181500000.0,230000.0,24,1998-12-20,182,75,73.0,Paris Saint Germain,16.0,Ligue 1,1.0,LW,7.0,NaN,2018-07-01,2024.0,18,France,1335.0,LW,10.0,Right,4,5,5,High/Low,Unique,Yes,349400000.0,"#Speedster, #Dribbler, #Acrobat, #Clinical finisher, #Co...","Quick Step +, Rapid, Flair, Trivela",97.0,90.0,80.0,92.0,36.0,78.0,78,94,73,86,84,93,80,69,71,92,97,97,93,93,82,90,88,88,77,83,64,38,93,83,84,88.0,26,34,32,13,5,7,11,6,NaN,90+3,90+3,90+3,91,91,91,91,91,89+3,89+3,89+3,89+3,81+3,81+3,81+3,89+3,68+3,63+3,63+3,63+3,68+3,63+3,54+3,54+3,54+3,63+3,18+3
1,239085,/player/239085/erling-haaland/240002,24.0,2.0,2023-09-22,E. Haaland,Erling Braut Haaland,ST,91,94,185000000.0,340000.0,22,2000-07-21,195,94,10.0,Manchester City,13.0,Premier League,1.0,ST,9.0,NaN,2022-07-01,2027.0,36,Norway,1352.0,ST,9.0,Left,3,3,5,High/Medium,Unique,Yes,356100000.0,"#Aerial threat, #Distance shooter, #Strength, #Clinical ...","Acrobatic +, Power Header, Quick Step",89.0,93.0,66.0,80.0,45.0,88.0,47,96,83,77,90,79,77,62,53,82,82,94,76,94,72,94,93,76,93,86,87,43,96,74,84,87.0,38,47,29,7,14,13,11,7,NaN,90+3,90+3,90+3,82,86,86,86,82,82+3,82+3,82+3,79+3,74+3,74+3,74+3,79+3,62+3,63+3,63+3,63+3,62+3,60+3,62+3,62+3,62+3,60+3,19+3
2,192985,/player/192985/kevin-de-bruyne/240002,24.0,2.0,2023-09-22,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,103000000.0,350000.0,32,1991-06-28,181,75,10.0,Manchester City,13.0,Premier League,1.0,SUB,17.0,NaN,2015-08-30,2025.0,7,Belgium,1325.0,CAM,7.0,Right,5,4,5,High/Medium,Unique,Yes,190600000.0,"#Dribbler, #Playmaker, #Distance shooter, #Crosser, #Com...","Pinged Pass +, Dead Ball, Incisive Pass, Long Ball Pass,...",72.0,88.0,94.0,87.0,65.0,78.0,95,85,55,94,83,86,92,83,94,92,72,72,74,92,78,92,72,88,74,92,75,66,88,95,83,88.0,66,70,53,15,13,5,10,13,NaN,83+3,83+3,83+3,87,88,88,88,87,89+2,89+2,89+2,88+3,90+1,90+1,90+1,88+3,79+3,80+3,80+3,80+3,79+3,75+3,70+3,70+3,70+3,75+3,21+3
3,158023,/player/158023/lionel-messi/240002,24.0,2.0,2023-09-22,L. Messi,Lionel Andrés Messi Cuccittini,"CF, CAM",90,90,41000000.0,23000.0,36,1987-06-24,169,67,112893.0,Inter Miami,39.0,Major League Soccer,1.0,RF,10.0,NaN,2023-07-16,2025.0,52,Argentina,1369.0,RW,10.0,Left,4,4,5,Low/Low,Unique,Yes,61500000.0,"#Dribbler, #Playmaker, #FK Specialist, #Acrobat, #Clinic...","Technical +, Finesse Shot, Dead Ball, Pinged Pass, Tiki ...",80.0,87.0,90.0,94.0,33.0,64.0,83,89,60,91,86,96,93,93,90,93,87,74,91,88,95,83,71,70,68,90,44,40,91,92,75,96.0,20,35,24,6,11,15,14,8,NaN,85+3,85+3,85+3,90,89,89,89,90,91-1,91-1,91-1,89+1,85+3,85+3,

,rows,n_dates,date,n_updates,label,expected_date
fifa_version,,,,,,
15,16182,1,2014-09-18,1,FIFA 15,2014-09-18
16,16706,1,2015-09-21,1,FIFA 16,2015-09-21
17,17596,1,2016-09-20,1,FIFA 17,2016-09-20
18,17954,1,2017-09-18,1,FIFA 18,2017-09-18
19,18086,1,2018-08-21,1,FIFA 19,2018-08-21
20,18483,1,2019-09-19,1,FIFA 20,2019-09-19
21,18892,1,2020-09-23,1,FIFA 21,2020-09-23
22,19239,1,2021-09-23,1,FIFA 22,2021-09-23
23,18533,1,2022-09-26,1,FIFA 23,2022-09-26


PASS - FIFA: one snapshot date per edition 
PASS - FIFA: editions are exactly 15–24 
PASS - FIFA: edition dates match CLAUDE.md section 7 


In [6]:
for name, col, note in [("male_teams.csv", "update_as_of", "Not used (CLAUDE.md: male_players.csv only)"),
                        ("male_coaches.csv", "dob", "Not used (CLAUDE.md: male_players.csv only)")]:
    p = config.RAW_FIFA / name
    if p.exists():
        df = pd.read_csv(p, low_memory=False)
        describe(name, df)
        add_inventory("FIFA", p, df, date_range(df[col], "dob" if col == "dob" else "date"), note)
        del df


=== male_teams.csv: 6,947 rows x 54 columns


,dtype,non_null,null_%,example
team_id,int64,6947,0.0,10
team_url,str,6947,0.0,/team/10/manchester-city/240002
fifa_version,int64,6947,0.0,24
fifa_update,int64,6947,0.0,2
update_as_of,str,6947,0.0,2023-09-22
team_name,str,6947,0.0,Manchester City
league_id,int64,6947,0.0,13
league_name,str,6947,0.0,Premier League
league_level,float64,6497,6.5,1.0
nationality_id,int64,6947,0.0,14


,team_id,team_url,fifa_version,fifa_update,update_as_of,team_name,league_id,league_name,league_level,nationality_id,nationality_name,overall,attack,midfield,defence,coach_id,home_stadium,rival_team,international_prestige,domestic_prestige,transfer_budget_eur,club_worth_eur,starting_xi_average_age,whole_team_average_age,captain,short_free_kick,long_free_kick,left_short_free_kick,right_short_free_kick,penalties,left_corner,right_corner,def_style,def_team_width,def_team_depth,def_defence_pressure,def_defence_aggression,def_defence_width,def_defence_defender_line,off_style,off_build_up_play,off_chance_creation,off_team_width,off_players_in_box,off_corners,off_free_kicks,build_up_play_speed,build_up_play_dribbling,build_up_play_passing,build_up_play_positioning,chance_creation_passing,chance_creation_crossing,chance_creation_shooting,chance_creation_positioning
0,10,/team/10/manchester-city/240002,24,2,2023-09-22,Manchester City,13,Premier League,1.0,14,England,85,87,86,83,455361.0,Etihad Stadium,11,10,10.0,NaN,4.600000e+09,26.91,25.14,188377.0,237692.0,246191.0,246191.0,237692.0,239085.0,246191.0,237692.0,Press after possession loss,71.0,61.0,NaN,NaN,NaN,NaN,NaN,Balanced,Balanced,81.0,6.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,243,/team/243/real-madrid/240002,24,2,2023-09-22,Real Madrid,53,La Liga,1.0,45,Spain,85,85,85,83,455800.0,Estadio Santiago Bernabéu,241,10,10.0,NaN,4.800000e+09,25.91,24.52,204963.0,197445.0,197445.0,239053.0,197445.0,243812.0,239053.0,239053.0,Balanced,51.0,71.0,NaN,NaN,NaN,NaN,NaN,Balanced,Balanced,51.0,6.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1318,/team/1318/england/240002,24,2,2023-09-22,England,78,Friendly International,NaN,14,England,85,86,83,83,474589.0,Wembley Stadium,1337,8,NaN,NaN,NaN,27.45,26.62,202126.0,205988.0,202126.0,205988.0,205988.0,202126.0,205988.0,205988.0,Press after possession loss,61.0,61.0,NaN,NaN,NaN,NaN,NaN,Slow build up,Possession,81.0,4.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21,/team/21/fc-bayern-munchen/240002,24,2,2023-09-22,FC Bayern München,19,Bundesliga,1.0,21,Germany,84,90,84,83,523937.0,Sanderson Park (FIFA 13 NEW generic),22,9,10.0,NaN,3.800000e+09,26.09,25.84,212622.0,222492.0,212622.0,212622.0,222492.0,202126.0,212622.0,222492.0,Pressure on heavy touch,51.0,81.0,NaN,NaN,NaN,NaN,NaN,Slow build up,Possession,71.0,8.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,241,/team/241/fc-barcelona/240002,24,2,2023-09-22,FC Barcelona,53,La Liga,1.0,45,Spain,84,84,84,83,184942.0,El Libertador (La Bombastico),243,10,10.0,NaN,4.300000e+09,27.00,23.94,192448.0,188545.0,186942.0,188545.0,233419.0,188545.0,228702.0,233419.0,Balanced,61.0,81.0,NaN,NaN,NaN,NaN,NaN,Slow build up,Possession,51.0,7.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== male_coaches.csv: 1,369 rows x 8 columns


,dtype,non_null,null_%,example
coach_id,int64,1369,0.0,24
coach_url,str,1369,0.0,/coach/24/david-unsworth/
short_name,str,1369,0.0,D. Unsworth
long_name,str,1369,0.0,David Unsworth
dob,str,1265,7.6,1973-10-16
nationality_name,str,1369,0.0,England
coach_face_url,str,957,30.1,https://cdn.sportmonks.com/images/soccer/players/18/50.png
nation_flag_url,str,1369,0.0,https://cdn.sofifa.net/flags/gb-eng.png


,coach_id,coach_url,short_name,long_name,dob,nationality_name,coach_face_url,nation_flag_url
0,24,/coach/24/david-unsworth/,D. Unsworth,David Unsworth,1973-10-16,England,NaN,https://cdn.sofifa.net/flags/gb-eng.png
1,50,/coach/50/steven-gerrard/,S. Gerrard,Steven Gerrard,1980-05-30,England,https://cdn.sportmonks.com/images/soccer/players/18/50.png,https://cdn.sofifa.net/flags/gb-eng.png
2,95,/coach/95/frank-lampard/,F. Lampard,Frank Lampard,1978-06-20,England,https://cdn.sportmonks.com/images/soccer/players/31/95.png,https://cdn.sofifa.net/flags/gb-eng.png
3,121,/coach/121/carl-hoefkens/,C. Hoefkens,Carl Hoefkens,1978-10-05,Belgium,https://cdn.sportmonks.com/images//soccer/coaches/25/121...,https://cdn.sofifa.net/flags/be.png
4,126,/coach/126/andy-dawson/,A. Dawson,Andy Dawson,1978-10-20,England,NaN,https://cdn.sofifa.net/flags/gb-eng.png


## 5. Transfermarkt (davidcariboo/player-scores)
List every table and its columns. Then confirm the columns notebook 02 depends on.

In [7]:
TM_DATE_COLS = {  # table -> (column, kind) used for the inventory date range
    "appearances": ("date", "date"), "games": ("date", "date"), "game_events": ("date", "date"),
    "game_lineups": ("date", "date"), "player_valuations": ("date", "date"),
    "transfers": ("transfer_date", "date"), "players": ("date_of_birth", "dob"),
    "clubs": ("last_season", "season"), "national_teams": ("last_season", "season"),
}
TM_NOTES = {
    "club_games": "No date column; join to games on game_id",
    "competitions": "Lookup table", "countries": "Lookup table",
}
TM_KEEP = {"players", "appearances", "competitions"}  # kept in memory for the checks below

tm_files = sorted(config.RAW_TM.glob("*.csv"))
print("Transfermarkt tables:", [p.stem for p in tm_files])
tm = {}
for p in tm_files:
    df = pd.read_csv(p, low_memory=False)
    describe(p.name, df)
    col, kind = TM_DATE_COLS.get(p.stem, (None, None))
    rng = date_range(df[col], kind) if col in df.columns else ""
    note = TM_NOTES.get(p.stem, "")
    if col in df.columns and kind == "date":
        mx = pd.to_datetime(df[col], errors="coerce").max()
        if mx > pd.Timestamp.today():
            n_future = int((pd.to_datetime(df[col], errors="coerce") > pd.Timestamp.today()).sum())
            note = f"{n_future:,} rows dated after today (max {mx:%Y-%m-%d})"
    add_inventory("Transfermarkt", p, df, rng, note)
    if p.stem in TM_KEEP:
        tm[p.stem] = df
    del df

Transfermarkt tables: ['appearances', 'club_games', 'clubs', 'competitions', 'countries', 'game_events', 'game_lineups', 'games', 'national_teams', 'player_valuations', 'players', 'transfers']



=== appearances.csv: 1,894,350 rows x 13 columns


,dtype,non_null,null_%,example
appearance_id,str,1894350,0.0,2231978_38004
game_id,int64,1894350,0.0,2231978
player_id,int64,1894350,0.0,38004
player_club_id,int64,1894350,0.0,853
player_current_club_id,int64,1894350,0.0,235
date,str,1894350,0.0,2012-07-03
player_name,str,1894348,0.0,Aurélien Joachim
competition_id,str,1894350,0.0,CLQ
yellow_cards,int64,1894350,0.0,0
red_cards,int64,1894350,0.0,0


,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90



=== club_games.csv: 177,916 rows x 11 columns


,dtype,non_null,null_%,example
game_id,int64,177916,0.0,1026846
club_id,int64,177916,0.0,3449
own_goals,int64,177916,0.0,2
own_position,float64,127162,28.5,3.0
own_manager_name,str,176228,0.9,Óscar Tabárez
opponent_id,int64,177916,0.0,3589
opponent_goals,int64,177916,0.0,1
opponent_position,float64,127162,28.5,13.0
opponent_manager_name,str,176228,0.9,Jung-moo Huh
hosting,str,177916,0.0,Home


... (27 outputs left out)

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,date_of_birth,sub_position,position,foot,height_in_cm,contract_expiration_date,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,1978-06-09 00:00:00,Centre-Forward,Attack,right,184.0,NaN,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrait/header/1...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose/profil/sp...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,1980-08-06 00:00:00,Goalkeeper,Goalkeeper,left,190.0,NaN,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrait/header/2...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfeller/profi...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,1981-01-30 00:00:00,Centre-Forward,Attack,NaN,NaN,NaN,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrait/header/6...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbatov/profil/...,GR1,Panthessalonikios Athlitikos Omilos Konstantinoupoliton,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,1978-05-08 00:00:00,Centre-Back,Defender,NaN,NaN,NaN,NaN,https://img.a.transfermarkt.technology/portrait/header/7...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/spieler/77,IT1,Juventus FC,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,1981-03-18 00:00:00,Goalkeeper,Goalkeeper,right,194.0,NaN,IFM,https://img.a.transfermarkt.technology/portrait/header/8...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/profil/spiele...,L1,Bayern Munich,100000.0,3000000.0



=== transfers.csv: 175,165 rows x 10 columns


,dtype,non_null,null_%,example
player_id,int64,175165,0.0,467994
transfer_date,str,175165,0.0,2030-06-30
transfer_season,str,175165,0.0,25/26
from_club_id,int64,175165,0.0,5621
to_club_id,int64,175165,0.0,749
from_club_name,str,175165,0.0,Reggiana
to_club_name,str,175165,0.0,FC Empoli
transfer_fee,float64,113639,35.1,0.0
market_value_in_eur,float64,107005,38.9,700000.0
player_name,str,175165,0.0,Luca Belardinelli


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,645842,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,150000.0,Chan-gi An
2,677470,2028-02-02,27/28,6505,3535,Gimcheon Sangmu,Ulsan HD,0.0,400000.0,Yool Heo
3,709155,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,325000.0,Jae-hyeok Oh
4,728133,2028-02-02,27/28,6505,35759,Gimcheon Sangmu,Bucheon FC,0.0,175000.0,Sang-hyeok Lee


In [8]:
present = {p.stem for p in tm_files}
missing_tables = [t for t in config.TM_REQUIRED_TABLES if t not in present]
check("TM: required tables present (players, player_valuations, appearances, clubs, competitions)",
      not missing_tables, f"missing: {missing_tables}" if missing_tables else "")

TM_REQUIRED_COLS = {
    "players": ["player_id", "date_of_birth", "name", "first_name", "last_name", "current_club_id"],
    "appearances": ["player_id", "date", "minutes_played", "competition_id", "player_club_id"],
    "competitions": ["competition_id", "name", "type", "country_name"],
}
for t, cols in TM_REQUIRED_COLS.items():
    miss = [c for c in cols if c not in tm[t].columns]
    check(f"TM {t}: columns {cols}", not miss, f"missing: {miss}" if miss else "")

# Other tables are not held in memory; re-check their headers cheaply.
for t, cols in {"player_valuations": ["player_id", "date", "market_value_in_eur"],
                "clubs": ["club_id", "name", "domestic_competition_id"]}.items():
    hdr = pd.read_csv(config.RAW_TM / f"{t}.csv", nrows=0).columns
    miss = [c for c in cols if c not in hdr]
    check(f"TM {t}: columns {cols}", not miss, f"missing: {miss}" if miss else "")

big5 = tm["competitions"].set_index("competition_id").reindex(config.BIG5_COMPETITIONS)[["name", "country_name", "type"]]
display(big5)
check("TM: Big-5 competition IDs (GB1, ES1, L1, IT1, FR1) all in competitions",
      big5["name"].notna().all())

dob = pd.to_datetime(tm["players"]["date_of_birth"], errors="coerce")
check("TM players: date_of_birth parses", True,
      f"{dob.notna().mean():.1%} of {len(dob):,} players have a parseable DOB")
mins = pd.to_numeric(tm["appearances"]["minutes_played"], errors="coerce")
check("TM appearances: minutes_played is numeric", mins.notna().all(),
      f"range {mins.min():.0f}–{mins.max():.0f}")

name_like = [c for c in tm["players"].columns if "name" in c.lower()]
print("Name-like columns in players:", name_like)
if not any("short" in c.lower() for c in name_like):
    FLAGS.append("Transfermarkt `players` has no separate short-name column (only `first_name`, "
                 "`last_name`, `name`). CLAUDE.md §8 (notebook 02) mentions TM 'full and short names'; "
                 "notebook 02 must decide what to compare against FIFA `short_name`.")

PASS - TM: required tables present (players, player_valuations, appearances, clubs, competitions) 
PASS - TM players: columns ['player_id', 'date_of_birth', 'name', 'first_name', 'last_name', 'current_club_id'] 
PASS - TM appearances: columns ['player_id', 'date', 'minutes_played', 'competition_id', 'player_club_id'] 
PASS - TM competitions: columns ['competition_id', 'name', 'type', 'country_name'] 
PASS - TM player_valuations: columns ['player_id', 'date', 'market_value_in_eur'] 
PASS - TM clubs: columns ['club_id', 'name', 'domestic_competition_id'] 


,name,country_name,type
competition_id,,,
GB1,premier-league,England,domestic_league
ES1,laliga,Spain,domestic_league
L1,bundesliga,Germany,domestic_league
IT1,serie-a,Italy,domestic_league
FR1,ligue-1,France,domestic_league


PASS - TM: Big-5 competition IDs (GB1, ES1, L1, IT1, FR1) all in competitions 
PASS - TM players: date_of_birth parses | 99.9% of 50,149 players have a parseable DOB
PASS - TM appearances: minutes_played is numeric | range 0–148
Name-like columns in players: ['first_name', 'last_name', 'name', 'agent_name', 'current_club_name']


In [9]:
# Earliest Big-5 appearance per league: a first look only; notebook 02 owns the full gap analysis.
a = tm["appearances"]
first_app = (a[a["competition_id"].isin(config.BIG5_COMPETITIONS)]
             .groupby("competition_id")["date"].agg(["min", "max", "size"]))
display(first_app)
late = first_app[pd.to_datetime(first_app["min"]) > pd.Timestamp(EDITION_DATES[15])]
if len(late):
    FLAGS.append("Big-5 appearance records start after the FIFA 15 edition date for: "
                 + ", ".join(f"{i} ({r['min']})" for i, r in late.iterrows()))
FLAGS.append("Transfermarkt Big-5 appearances start at "
             + ", ".join(f"{i} {r['min']}" for i, r in first_app.iterrows())
             + ". Notebook 02 must check coverage across all leagues and record it in DECISIONS.md.")
del a, tm

,min,max,size
competition_id,,,
ES1,2012-08-18,2026-05-24,153877
FR1,2012-08-10,2026-05-17,143395
GB1,2012-08-18,2026-05-24,149751
IT1,2012-08-25,2026-05-24,155187
L1,2012-08-24,2026-05-16,124613


## 6. FIFA Men's World Ranking (lucasyukioimafuko/fifa-mens-world-ranking)
CLAUDE.md §5: coverage must span Sept 2014 to Sept 2023. Use **rank**, never points.

In [10]:
rank = pd.read_csv(config.RANKING_FILE)
describe(config.RANKING_FILE.name, rank)

periods = rank.groupby(["date", "semester"]).agg(teams=("team", "size"), max_rank=("rank", "max")).reset_index()
print("Ranking periods (date = year):")
display(periods[periods["date"].between(2013, 2024)])

need = {(y, s) for y in range(2014, 2024) for s in (1, 2)}
have = set(zip(periods["date"], periods["semester"]))
gap = sorted(need - have)
check("Ranking: covers every half-year 2014–2023", not gap, f"missing periods: {gap}" if gap else
      f"{periods['date'].min()} S{periods.iloc[0]['semester']} to {periods['date'].max()} S{periods.iloc[-1]['semester']}")

for c in ["rank", "team"]:
    check(f"Ranking: column `{c}` exists", c in rank.columns)

FLAGS.append("FIFA ranking has year + semester (1 or 2) only, no exact publication dates: "
             f"{len(periods)} periods, two per year. 'Latest ranking date on or before the edition date' "
             "(CLAUDE.md §6) cannot be applied literally; notebook 04 must decide how to map a "
             "semester to a date and record it in DECISIONS.md.")
if not any("confed" in c.lower() for c in rank.columns):
    FLAGS.append("FIFA ranking has no confederation column. Transfermarkt `national_teams.csv` and "
                 "`countries.csv` have one, which notebook 04 could join for chart 9.")
dup = rank.duplicated(["date", "semester", "rank"]).sum()
if dup:
    FLAGS.append(f"FIFA ranking has {dup:,} tied (date, semester, rank) rows across all years; "
                 f"{rank[rank['date'].between(2014, 2023)].duplicated(['date', 'semester', 'rank']).sum():,} "
                 "of them in 2014–2023.")

add_inventory("FIFA ranking", config.RANKING_FILE, rank,
              f"{periods['date'].min()} S{periods.iloc[0]['semester']} to "
              f"{periods['date'].max()} S{periods.iloc[-1]['semester']} (year + semester only)",
              f"{len(periods)} half-yearly periods; rank + points, no confederation")


=== fifa_mens_rank.csv: 13,130 rows x 8 columns


,dtype,non_null,null_%,example
date,int64,13130,0.0,2024
semester,int64,13130,0.0,2
rank,int64,13130,0.0,1
team,str,13130,0.0,Argentina
acronym,str,13130,0.0,ARG
total.points,float64,13130,0.0,1867.25
previous.points,float64,13130,0.0,1883.5
diff.points,float64,13130,0.0,-16.25


,date,semester,rank,team,acronym,total.points,previous.points,diff.points
0,2024,2,1,Argentina,ARG,1867.25,1883.50,-16.25
1,2024,2,2,France,FRA,1859.78,1859.85,-0.07
2,2024,2,3,Spain,ESP,1853.27,1844.33,8.94
3,2024,2,4,England,ENG,1813.81,1807.83,5.98
4,2024,2,5,Brazil,BRA,1775.85,1784.37,-8.52


Ranking periods (date = year):

,date,semester,teams,max_rank
41,2013,1,209,207
42,2013,2,209,207
43,2014,1,209,208
44,2014,2,209,209
45,2015,1,209,209
46,2015,2,209,204
47,2016,1,210,205
48,2016,2,211,205
49,2017,1,211,206
50,2017,2,211,206


PASS - Ranking: covers every half-year 2014–2023 | 1992 S2 to 2024 S2
PASS - Ranking: column `rank` exists 
PASS - Ranking: column `team` exists 


## 7. ClubElo, one snapshot per edition date
`soccerdata.ClubElo().read_by_date(date)` for each of the 10 `update_as_of` dates. Results are cached
to `data/raw/clubelo/clubelo_YYYY-MM-DD.csv`. Cached dates are not re-downloaded.

**While api.clubelo.com is down** (since 4 Nov 2026, see `DECISIONS.md`): for each of the 4 cohort entry dates the API
doesn't return, a snapshot is built from the approved mirror (Kaggle *Club Football Match Data (2000–2025)* by adamgbor,
Elo data from ClubElo) and saved as `clubelo_mirror_YYYY-MM-DD.csv`. Only the entry dates are needed (notebook 03).
The other 6 editions are reported, not required.

In [11]:
logging.getLogger("soccerdata").setLevel(logging.CRITICAL)  # we report failures ourselves

elo_status, elo_frames = [], {}
for ed, date in EDITION_DATES.items():
    try:
        df, source = clubelo.fetch_snapshot(date)
        elo_frames[date] = df
        elo_status.append({"Edition": config.EDITION_LABELS[ed], "Date": date, "Status": source,
                           "Clubs": len(df), "Error": ""})
    except Exception as e:
        elo_status.append({"Edition": config.EDITION_LABELS[ed], "Date": date, "Status": "FAILED",
                           "Clubs": None, "Error": f"{type(e).__name__}: {e}"[:150]})
elo_status = pd.DataFrame(elo_status)
display(elo_status)

n_ok = int((elo_status["Status"] != "FAILED").sum())
print(f"API snapshots available: {n_ok}/{len(EDITION_DATES)}")

[11/04/26 20:40:50] INFO     No custom team name replacements found. You can configure these in       ]8;id=16300779;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=16300780;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/sahilmohanty/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=16300786;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=16300787;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_config.py#189\189]8;;\
                             /Users/sahilmohanty/soccerdata/config/league_dict.json.                               

                    INFO     Saving cached data to                                                   ]8;id=16300794;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=16300795;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py#300\300]8;;\
                             /Users/sahilmohanty/Documents/career-mode-promise/data/raw/clubelo/_soc               
                             cerdata_cache                                                                         

[2026-11-04 20:40:50] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib


                    INFO     Successfully loaded TLS library:                                      ]8;id=16300802;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=16300803;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.1                 
                             1/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib                 

                    ERROR    Error while scraping http://api.clubelo.com/2014-09-18. Retrying...     ]8;id=16300809;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=16300810;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py#593\593]8;;\
                             (attempt 1 of 5).                                                                     
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11               
                             /site-packages/soccerdata/_common.py", line 573, in _download_and_save                
                                 response.raise_for_status()                                                       
                               File                                                                                
                             "/Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11               
                             /site-packages/tls_requests/models/response.py", line 207, in                         
                             raise_for_status                                                                      
                                 raise HTTPError(                                                                  
                             tls_requests.exceptions.HTTPError: 502 Server Error: Bad Gateway for                  
                             url: http://api.clubelo.com/2014-09-18                                                

... (56 outputs left out)

[11/04/26 20:41:16] ERROR    Error while scraping http://api.clubelo.com/2023-09-22. Retrying...     ]8;id=16301151;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=16301152;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py#593\593]8;;\
                             (attempt 4 of 5).                                                                     
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11               
                             /site-packages/soccerdata/_common.py", line 573, in _download_and_save                
                                 response.raise_for_status()                                                       
                               File                                                                                
                             "/Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11               
                             /site-packages/tls_requests/models/response.py", line 207, in                         
                             raise_for_status                                                                      
                                 raise HTTPError(                                                                  
                             tls_requests.exceptions.HTTPError: 502 Server Error: Bad Gateway for                  
                             url: http://api.clubelo.com/2023-09-22                                                

                    ERROR    Error while scraping http://api.clubelo.com/2023-09-22. Retrying...     ]8;id=16301157;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=16301158;file:///Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11/site-packages/soccerdata/_common.py#593\593]8;;\
                             (attempt 5 of 5).                                                                     
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11               
                             /site-packages/soccerdata/_common.py", line 573, in _download_and_save                
                                 response.raise_for_status()                                                       
                               File                                                                                
                             "/Users/sahilmohanty/Documents/career-mode-promise/.venv/lib/python3.11               
                             /site-packages/tls_requests/models/response.py", line 207, in                         
                             raise_for_status                                                                      
                                 raise HTTPError(                                                                  
                             tls_requests.exceptions.HTTPError: 502 Server Error: Bad Gateway for                  
                             url: http://api.clubelo.com/2023-09-22                                                

,Edition,Date,Status,Clubs,Error
0,FIFA 15,2014-09-18,FAILED,None,ConnectionError: Could not download http://api...
1,FIFA 16,2015-09-21,FAILED,None,ConnectionError: Could not download http://api...
2,FIFA 17,2016-09-20,FAILED,None,ConnectionError: Could not download http://api...
3,FIFA 18,2017-09-18,FAILED,None,ConnectionError: Could not download http://api...
4,FIFA 19,2018-08-21,FAILED,None,ConnectionError: Could not download http://api...
5,FIFA 20,2019-09-19,FAILED,None,ConnectionError: Could not download http://api...
6,FIFA 21,2020-09-23,FAILED,None,ConnectionError: Could not download http://api...
7,FIFA 22,2021-09-23,FAILED,None,ConnectionError: Could not download http://api...
8,FIFA 23,2022-09-26,FAILED,None,ConnectionError: Could not download http://api...
9,FC 24,2023-09-22,FAILED,None,ConnectionError: Could not download http://api...


API snapshots available: 0/10

In [12]:
# Mirror fallback for the cohort entry dates the API did not return
entry_dates = [EDITION_DATES[e] for e in config.COHORT_EDITIONS]
need_mirror = [d for d in entry_dates if d not in elo_frames]
mirror_status = []
if need_mirror:
    m_ratings = pd.read_csv(config.RAW_CLUBELO_MIRROR / "EloRatings.csv", parse_dates=["date"])
    m_matches = pd.read_csv(config.RAW_CLUBELO_MIRROR / "Matches.csv", parse_dates=["MatchDate"],
                            usecols=["Division", "MatchDate", "HomeTeam", "AwayTeam", "HomeElo", "AwayElo"])
    describe("club_elo_mirror/EloRatings.csv", m_ratings)
    add_inventory("ClubElo mirror", config.RAW_CLUBELO_MIRROR / "EloRatings.csv", m_ratings,
                  date_range(m_ratings["date"]), "ClubElo snapshots on the 1st and 15th of each month; ClubElo data through 2025-06-01")
    add_inventory("ClubElo mirror", config.RAW_CLUBELO_MIRROR / "Matches.csv", m_matches,
                  date_range(m_matches["MatchDate"]), "football-data.co.uk matches; used only for the level column")
    for d in need_mirror:
        p = clubelo.mirror_cache_path(d)
        snap = pd.read_csv(p) if p.exists() else clubelo.build_mirror_snapshot(d, m_ratings, m_matches)
        if not p.exists():
            snap.to_csv(p, index=False, encoding="utf-8")
        ok = len(snap) >= 300 and not snap["team"].duplicated().any()
        mirror_status.append({"Date": d, "Rating date": snap["rating_date"].iat[0], "Clubs": len(snap),
                              "Countries": snap["country"].nunique(), "Duplicate clubs": int(snap["team"].duplicated().sum()),
                              "Top 3": ", ".join(snap.nlargest(3, "elo")["team"]), "OK": ok})
        elo_frames[d] = snap
    mirror_status = pd.DataFrame(mirror_status)
    display(mirror_status)
    check("ClubElo mirror snapshots: 300+ clubs and no duplicate clubs", bool(mirror_status["OK"].all()))
covered = [d for d in entry_dates if d in elo_frames]
check("ClubElo: a snapshot (API or mirror) for each of the 4 cohort entry dates", len(covered) == len(entry_dates),
      f"{len(covered)}/{len(entry_dates)}; mirror used for {need_mirror if need_mirror else 'none'}")


=== club_elo_mirror/EloRatings.csv: 273,972 rows x 4 columns

,dtype,non_null,null_%,example
date,datetime64[us],273972,0.0,2000-07-01 00:00:00
club,str,273972,0.0,Aachen
country,str,273972,0.0,GER
elo,float64,273972,0.0,1453.6


,date,club,country,elo
0,2000-07-01,Aachen,GER,1453.60
1,2000-07-01,Aalborg,DEN,1482.61
2,2000-07-01,Aalst,BEL,1337.53
3,2000-07-01,Aarhus,DEN,1381.46
4,2000-07-01,Aberdeen,SCO,1360.43


,Date,Rating date,Clubs,Countries,Duplicate clubs,Top 3,OK
0,2014-09-18,2014-09-15,415,19,0,"Real Madrid, Barcelona, Bayern Munich",True
1,2015-09-21,2015-09-15,406,19,0,"Barcelona, Real Madrid, Bayern Munich",True
2,2016-09-20,2016-09-15,409,19,0,"Real Madrid, Barcelona, Bayern Munich",True
3,2017-09-18,2017-09-15,408,19,0,"Real Madrid, Barcelona, Bayern Munich",True


PASS

-

ClubElo mirror snapshots: 300+ clubs and no duplicate clubs

PASS

-

ClubElo: a snapshot (API or mirror) for each of the 4 cohort entry dates

| 4/4; mirror used for ['2014-09-18', '2015-09-21', '2016-09-20', '2017-09-18']

In [13]:
if elo_frames:
    first = next(iter(elo_frames))
    describe(f"clubelo_{first}.csv", elo_frames[first])
    allelo = pd.concat(elo_frames.values(), ignore_index=True)
    add_inventory("ClubElo", config.RAW_CLUBELO, allelo,
                  f"{len(elo_frames)} snapshots: {min(elo_frames)} to {max(elo_frames)}",
                  "one file per edition date, clubelo_YYYY-MM-DD.csv; rows summed over snapshots")
    for d, df in elo_frames.items():
        print(d, "clubs:", len(df), "| countries:", df["country"].nunique(), "| levels:",
              sorted(df["level"].dropna().unique().tolist()))
failed = elo_status[(elo_status["Status"] == "FAILED") & ~elo_status["Date"].isin(entry_dates)]
if len(failed):
    INVENTORY.append({"Source": "ClubElo", "File": "clubelo/ (missing snapshots)", "Rows": None,
                      "Columns": None, "Date range": ", ".join(failed["Date"]),
                      "Notes": "NOT DOWNLOADED: " + failed["Error"].iloc[0]})
    FLAGS.append(f"ClubElo API: {len(failed)} non-entry edition snapshots not downloaded "
                 f"({failed['Error'].iloc[0]}). Not needed: club strength is only measured at the 4 cohort entry dates.")
if need_mirror:
    FLAGS.append(f"ClubElo: the API did not return the cohort entry dates {need_mirror}; mirror snapshots used "
                 "(Club Football Match Data (2000–2025), Kaggle, by adamgbor; Elo data from ClubElo).")


=== clubelo_2014-09-18.csv: 415 rows x 12 columns

,dtype,non_null,null_%,example
snapshot_date,str,415,0.0,2014-09-18
team,str,415,0.0,Real Madrid
rank,float64,415,0.0,1.0
country,str,415,0.0,ESP
level,float64,300,27.7,1.0
elo,float64,415,0.0,2040.99
from,float64,0,100.0,None
to,float64,0,100.0,None
league,str,300,27.7,ESP_1
source,str,415,0.0,mirror: Kaggle adamgbor/club-football-match-data-2000-20...


,snapshot_date,team,rank,country,level,elo,from,to,league,source,rating_date,division
0,2014-09-18,Real Madrid,1.0,ESP,1.0,2040.99,NaN,NaN,ESP_1,mirror: Kaggle adamgbor/club-football-match-data-2000-20...,2014-09-15,SP1
1,2014-09-18,Barcelona,2.0,ESP,1.0,2030.61,NaN,NaN,ESP_1,mirror: Kaggle adamgbor/club-football-match-data-2000-20...,2014-09-15,SP1
2,2014-09-18,Bayern Munich,3.0,GER,1.0,2029.84,NaN,NaN,GER_1,mirror: Kaggle adamgbor/club-football-match-data-2000-20...,2014-09-15,D1
3,2014-09-18,Ath Madrid,4.0,ESP,1.0,1979.83,NaN,NaN,ESP_1,mirror: Kaggle adamgbor/club-football-match-data-2000-20...,2014-09-15,SP1
4,2014-09-18,Man City,5.0,ENG,1.0,1947.30,NaN,NaN,ENG_1,mirror: Kaggle adamgbor/club-football-match-data-2000-20...,2014-09-15,E0


2014-09-18

clubs:

415

... (21 outputs left out)

| countries:

19

| levels:

[1.0, 2.0]

## 8. Inventory, written to `reports/data_quality.md`

In [14]:
inv = pd.DataFrame(INVENTORY)
inv["Rows"] = inv["Rows"].astype("Int64")
inv["Columns"] = inv["Columns"].astype("Int64")
display(inv)

checks = pd.DataFrame(CHECKS)
body = ("### Inventory\n\n" + dq.df_to_md(inv) +
        "\n\n### ClubElo download status\n\n" + dq.df_to_md(elo_status) +
        "\n\n### Checks\n\n" + dq.df_to_md(checks) +
        "\n\n### Flags for later notebooks\n\n" + "\n".join(f"- {f}" for f in FLAGS))
dq.write_section("00", "00 Setup and download: source inventory", body)
print("Written to", config.DATA_QUALITY.relative_to(ROOT))

,Source,File,Rows,Columns,Date range,Notes
0,FIFA,fifa/male_players.csv,180021,109,2014-09-18 to 2023-09-22,"10 editions (15–24), one launch snapshot each;..."
1,FIFA,fifa/male_teams.csv,6947,54,2014-09-18 to 2023-09-22,Not used (CLAUDE.md: male_players.csv only)
2,FIFA,fifa/male_coaches.csv,1369,8,DOB 1945-07-29 to 2005-01-11,Not used (CLAUDE.md: male_players.csv only)
3,Transfermarkt,transfermarkt/appearances.csv,1894350,13,2012-07-03 to 2026-06-28,
4,Transfermarkt,transfermarkt/club_games.csv,177916,11,,No date column; join to games on game_id
5,Transfermarkt,transfermarkt/clubs.csv,796,17,seasons 2012–2025,
6,Transfermarkt,transfermarkt/competitions.csv,65,11,,Lookup table
7,Transfermarkt,transfermarkt/countries.csv,124,8,,Lookup table
8,Transfermarkt,transfermarkt/game_events.csv,1274469,11,2006-06-09 to 2026-07-06,
9,Transfermarkt,transfermarkt/game_lineups.csv,3179016,10,2013-07-02 to 2026-07-06,


Written to

reports/data_quality.md

## 9. Summary

In [15]:
display(checks)
print("\nFlags for later notebooks:")
for f in FLAGS:
    print(" -", f)
failed_checks = checks[checks["Result"] == "FAIL"]
assert failed_checks.empty, f"{len(failed_checks)} check(s) failed: {failed_checks['Check'].tolist()}"

,Check,Result,Detail
0,All Kaggle source files present,PASS,
1,FIFA: one snapshot date per edition,PASS,
2,FIFA: editions are exactly 15–24,PASS,
3,FIFA: edition dates match CLAUDE.md section 7,PASS,
4,"TM: required tables present (players, player_v...",PASS,
5,"TM players: columns ['player_id', 'date_of_bir...",PASS,
6,"TM appearances: columns ['player_id', 'date', ...",PASS,
7,"TM competitions: columns ['competition_id', 'n...",PASS,
8,"TM player_valuations: columns ['player_id', 'd...",PASS,
9,"TM clubs: columns ['club_id', 'name', 'domesti...",PASS,



Flags for later notebooks:

 -

Transfermarkt `players` has no separate short-name column (only `first_name`, `last_name`, `name`). CLAUDE.md §8 (notebook 02) mentions TM 'full and short names'; notebook 02 must decide what to compare against FIFA `short_name`.

 -

Transfermarkt Big-5 appearances start at ES1 2012-08-18, FR1 2012-08-10, GB1 2012-08-18, IT1 2012-08-25, L1 2012-08-24. Notebook 02 must check coverage across all leagues and record it in DECISIONS.md.

... (6 outputs left out)

 -

ClubElo API: 6 non-entry edition snapshots not downloaded (ConnectionError: Could not download http://api.clubelo.com/2018-08-21.). Not needed: club strength is only measured at the 4 cohort entry dates.

 -

ClubElo: the API did not return the cohort entry dates ['2014-09-18', '2015-09-21', '2016-09-20', '2017-09-18']; mirror snapshots used (Club Football Match Data (2000–2025), Kaggle, by adamgbor; Elo data from ClubElo).